In [7]:
!pip install -q langchain langchain_community langchain-anthropic faiss-cpu langchain_huggingface unstructured

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 37.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 586.9/586.9 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.0/298.0 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.6/114.6 kB 9.8 MB/s eta 0:00:00


In [2]:
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_community.vectorstores import FAISS

from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_anthropic import ChatAnthropic

import os

In [3]:
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "YOUR_HUGGINGFACE_TOKEN"
os.environ["ANTHROPIC_API_KEY"] = "YOUR_ANTHROPIC_API_KEY"

In [4]:
def get_retriever(URL: str):
    # Carregar documentos da URL
    loader = UnstructuredURLLoader([URL],
                                   show_progress_bar=True)
    documents = loader.load()

    # Dividir documentos em partes menores
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    docs = text_splitter.split_documents(documents)

    # Criar embeddings
    embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

    # Criar um banco de dados FAISS a partir dos documentos
    db = FAISS.from_documents(docs, embeddings)

    # Criar um retriever a partir do banco de dados
    retriever = db.as_retriever()

    return retriever

In [8]:
URL = "https://thebentzgroup.com/br/?contentAdId=186603899&mediaSource=b1_outbrainrtb&rr=20241129181746&domain=ntp%2Emsn%2Ecom&placementname=edgechrntp-river-2&adgroupid=68045431&pageurl={pageurl}&AdTitle=Brasileiros+com+mais+de+35+anos+podem+aproveitar+oportunidades+de+renda+gratificantes.&zpbid=78926_2d6ef242-ae7e-11ef-b0d7-a07dfed7c987"
retriever_instance = get_retriever(URL)

100%|██████████| 1/1 [00:02<00:00,  2.65s/it]
/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

... (2 outputs left out)

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [10]:
model="claude-3-5-sonnet-20241022"
temperature=0.1
max_tokens=512
llm = ChatAnthropic(model=model,
                    temperature=temperature,
                    max_tokens=max_tokens)

In [11]:
def response(query: str, retriever, llm):

    context = retriever.invoke(query)
    # Definindo o prompt do sistema
    system_prompt = f"""
    Você é um assistente de IA projetado para responder perguntas com
    base no contexto fornecido. Sua tarefa é ler o contexto fornecido
    cuidadosamente e usá-lo para responder à consulta do usuário.

    Aqui está o contexto que você deve usar para formular sua resposta:
    {context}

    A pergunta do usuário é:
    {query}

    Siga estas diretrizes ao formular sua resposta:
    1. Leia e entenda cuidadosamente o contexto e a consulta.
    2. Use apenas as informações fornecidas no contexto para responder à consulta.
    3. Se a resposta à consulta não for encontrada no contexto, ou se você não tiver certeza, simplesmente diga que não sabe.
    4. Mantenha sua resposta concisa e direta.
    5. Não inclua nenhuma informação que não seja diretamente relevante para responder à consulta.
    6. Não invente ou infira informações que não estejam explicitamente declaradas no contexto.

    Lembre-se de responder em português.
    """

    # Invocando o modelo com o prompt
    response = llm.invoke([SystemMessage(content=system_prompt),
                            HumanMessage(content=query)])

    # Retornando o conteúdo da resposta
    return response.content if hasattr(response, 'content') else response

In [12]:
%%time
query = "do que se trata a noticia?"
response(query, retriever_instance, llm)

CPU times: user 58.4 ms, sys: 4.39 ms, total: 62.8 ms
Wall time: 5.67 s


'Com base no contexto fornecido, trata-se de informações sobre o The Bentz Group, uma empresa de consultoria de gestão boutique que opera mundialmente. A empresa:\n\n- Tem mais de 18 anos de atividade\n- Completou mais de 400 projetos\n- Afirma ter 100% de experiência positiva com clientes\n- Foca em executar estratégias, otimizar processos, aumentar produtividade e reduzir custos\n- Trabalha em parceria com clientes para transformar seus negócios\n- É liderada pelo CEO Rowan Prescott\n\nA empresa se apresenta como uma consultoria boutique premiada que trabalha com melhoria de desempenho organizacional, mantendo valores de serviço, excelência, integridade e aprendizado.'

In [13]:
%%time
query = "Faça um resumo do assunto"
response(query, retriever_instance, llm)

CPU times: user 47.5 ms, sys: 3.02 ms, total: 50.6 ms
Wall time: 4.85 s


'Com base no contexto fornecido, o The Bentz Group é uma empresa de consultoria de gestão boutique premiada que atua globalmente. Eis os principais pontos:\n\nCaracterísticas principais:\n- 18+ anos de atividade\n- 400+ projetos concluídos\n- 100% de experiência positiva com clientes\n\nA empresa se concentra em:\n- Executar estratégias\n- Otimizar processos\n- Aumentar produtividade\n- Reduzir custos\n\nSua abordagem envolve:\n- Parceria com clientes motivados a transformar seus negócios\n- Trabalho colaborativo com membros-chave das equipes\n- Compromisso com resultados excepcionais\n- Foco em valores como serviço, excelência, integridade e aprendizado\n\nSegundo o CEO Rowan Prescott, a empresa se destaca como uma consultoria boutique de alto nível, mantendo forte profissionalismo em sua equipe. O texto apresenta diversos depoimentos positivos de clientes, que elogiam a qualidade dos serviços e a capacidade da equipe em gerar soluções efetivas.'